## Compare Embedding Layer and Linear Layer

In [1]:
import torch

In [2]:
idx = torch.tensor([0, 2, 3, 1]) # vocab 4
num_idx = max(idx)+1 # vocab 4
out_dim = 5 # 5 dim embedding

In [3]:
torch.manual_seed(123)
embedding = torch.nn.Embedding(num_idx, out_dim)
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

In [4]:
embedding(idx)

tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

In [5]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

In [6]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

In [7]:
linear.weight = torch.nn.Parameter(embedding.weight.T)
linear.weight

Parameter containing:
tensor([[ 0.3374,  1.3010,  0.6957, -2.8400],
        [-0.1778,  1.2753, -1.8061, -0.7849],
        [-0.3035, -0.2010, -1.1589, -1.4096],
        [-0.5880, -0.1606,  0.3255, -0.4076],
        [ 1.5810, -0.4015, -0.6315,  0.7953]], requires_grad=True)

In [8]:
linear(onehot.float())

tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

In [9]:
# Linear use multiply. Embedding use look up

### what's the difference between nn.Linear and torch.nn.Embedding

**`nn.Linear`** — affine transform `y = xW^T + b` on float feature vectors.
- Weight `[out_features, in_features]` + bias `[out_features]`
- Input: floats, shape `(..., in_features)` → output `(..., out_features)`
- Every input feature interacts with all outputs (dense matmul)

**`nn.Embedding`** — a lookup table: integer index → learned vector.
- Weight `[num_embeddings, embedding_dim]`, **no bias**
- Input: integer indices, shape `(...)` → output `(..., embedding_dim)` (adds a dim)
- Only the rows for the given indices are used (sparse lookup), supports `sparse=True` gradients

It's mathematically equivalent to `one_hot(idx) @ W` where `W` is a `Linear(in_features=num_embeddings, out_features=embedding_dim, bias=False)` — useful when `num_embeddings` is huge and inputs are categorical IDs (words, tokens, user IDs).

It's not a matrix multiply — `Embedding` **indexes rows**. `torch.tensor([3])` has shape `(1,)`, so it selects row 3 of the weight matrix `[6, 3]`, giving output shape `(1, 3)`.

```python
embedding_layer.weight.shape        # (6, 3)
embedding_layer.weight[3]           # [-0.4015, 0.9666, -1.1481]
embedding_layer(torch.tensor([3]))  # same values, shape (1, 3)
```

The leading dims of the input are treated as batch dims and preserved:

| input shape | output shape |
|---|---|
| `(1,)` | `(1, 3)` |
| `(4,)` | `(4, 3)` |
| `(2, 5)` | `(2, 5, 3)` |

The "equivalent" matmul would be `one_hot(3, 6) @ W` → `(1,6) @ (6,3) = (1,3)`, which is why it *behaves* like a 1×6 times 6×3 product.

---

